## Task 01. Loading data

First, Import all required modules

In [0]:
# importing spark session
# Databricks already provides SparkSession as "spark"
# from pyspark.sql import SparkSession   # commenting this

# data visualization modules 
import matplotlib.pyplot as plt
import plotly.express as px 

# pandas module 
import pandas as pd

# pyspark SQL functions 
from pyspark.sql.functions import col, when, count, udf

# pyspark data preprocessing modules
from pyspark.ml.feature import Imputer, StringIndexer, VectorAssembler, StandardScaler

# pyspark data modeling and model evaluation modules
from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator


### Building Spark Session
Creating a Spark Session, we can access Spark's data processing functionality and perform operations on large datasets in a distributed manner

In [0]:
# Databricks provides spark session
# spark = SparkSession.builder.appName("Customer_Churn_Prediction").getOrCreate()
spark

In [0]:
# Loading our data
data = spark.read.csv("/Workspace/Users/praz.vj@gmail.com/data-science-projects/01-ml-pyspark-customer-churn/data/dataset.csv", header=True, inferSchema=True)

In [0]:
# Print the data schema to check out the data types
data.printSchema()

In [0]:
# Get the data dimension - number of rows and columns
print("Number of rows:", data.count())
print("Number of columns:", len(data.columns))
display(data.columns)

### Task Summary
#### Task Goal
In this task, we set up the environment for our project, ran the Spark session, and loaded the customer dataset. And we checked the data types and data dimensions in our PySpark DataFrame.

#### Key Takeaways

* Use `data.printSchema()` to check the data types of each column in a PySpark DataFrame.
* Use `data.count()` to count the total number of rows in a PySpark DataFrame.
* Use `data.columns` to get a list of all the column names in a PySpark DataFrame.

## Task 02. Exploratory Data Analysis

* Distribution Analysis
* Correlation Analysis
* Univariate Analysis
* Finding Missing Values

In [0]:
# Let's define some lists to store different column names with different data types.

data.show(5)

In [0]:
data.dtypes

In [0]:
# Let's get all the numerical features and store them into a pandas dataframe.

numerical_columns = [name for name, typ in data.dtypes if typ=='double' or typ=='int']
numerical_columns

In [0]:
categorical_columns = [name for name, typ in data.dtypes if typ=='string']
categorical_columns

In [0]:
data.select(numerical_columns).show()

In [0]:
# Let's get all the numerical features and store them into a pandas dataframe.
df = data.select(numerical_columns).toPandas()
df.head()

In [0]:
# Let's create histograms to analyse the distribution of our numerical columns.

fig = plt.figure(figsize=(15,10))
ax = fig.gca()
df.hist(ax=ax, bins=20)
plt.show

In [0]:
# the tenure histogram looks odd. Almost all the values are under 100, but the x-axis is spread over 400
# this means, there must be some outlier values
# when we look at the statistics, we can see that the max value is 458, but the mean is at 32.
# we will get rid of this outlier in the pre-processing step
df.tenure.describe()

In [0]:
# Let's generate the correlation matrix
df.corr()

In [0]:
# Let's check the unique value count per each categorical variables

categorical_columns

In [0]:
data.groupBy("Contract").count().show()

In [0]:
for column in categorical_columns:
    data.groupBy(column).count().show()

In [0]:
# Let's find number of null values in all of our dataframe columns
for column in data.columns:
    data.select(count(when(col(column).isNull(),column)).alias(column)).show()

### Task Summary
#### Task Goal
In this task, we explored the dataset to better understand its columns and distributions. We identified outliers and missing values in numerical columns and plotted visualizations to detect any patterns or trends. Additionally, we checked for correlations between columns to understand their relationship and guide our modeling decisions.

#### Key Takeaways

* The first step in modeling our data is data exploration, where we analyze and understand our data. This includes identifying outliers, missing values, and interesting trends that can help us build better models.

* To visualize our data, we may need to convert the PySpark DataFrame into a Pandas DataFrame using the `toPandas()` method. This can allow us to use various data visualization tools available in Python.

* Histograms are useful to identify outliers in numerical columns. They show the distribution of values in a column and can help us understand if there are any unusual values or patterns that require further investigation.

### Task 03. Preprocess and Clean

* Handling the missing values
* Removing the outliers

Handling the missing values depend on the type of columns
* for categorical columns, the most common tecnique is to fill in with most frequent values or using a classification model to predict the missing value
* for numerical columns, we can inject the average value of the column. We use the preprocessing module in Pyspark called `imputer`

#### Handling the missing values

In [0]:
# Let's create a list of column names with missing values

columns_with_missing_values = [column for column in data.columns if data.select(count(when(col(column).isNull(),column))).collect()[0][0] > 0]
columns_with_missing_values

In [0]:
# Creating our Imputer
imputer = Imputer(strategy="mean", 
                  inputCols=columns_with_missing_values, 
                  outputCols=columns_with_missing_values)
imputer_model = imputer.fit(data)
data = imputer_model.transform(data)

for column in columns_with_missing_values:
    data.select(count(when(col(column).isNull(),column)).alias(column)).show()


#### Removing the outliers
Let's find the customer with the tenure higher than 100

In [0]:
data.select("*").where(data.tenure > 100).show()

In [0]:
print(data.count())
data = data.filter(data.tenure < 100)
print(data.count())


#### **Task 4 - Feature Preparation**
- Numerical Features 
    - Vector Assembling
    - Numerical Scaling
- Categorical Features
    - String Indexing
    - Vector Assembling

- Combining the numerical and categorical feature vectors




**Feature Preparation - Numerical Features** <br>

`Vector Assembling --> Standard Scaling` <br>

**Vector Assembling** <br>
To apply our machine learning model we need to combine all of our numerical and categorical features into vectors. For now let's create a feature vector for our numerical columns._

In [0]:
numerical_vector_assembler = VectorAssembler(
    inputCols=numerical_columns, 
    outputCol="numerical_features_vector")

data=numerical_vector_assembler.transform(data)
data.show()

In [0]:
# Numerical scaling

scaler = StandardScaler(
    inputCol="numerical_features_vector", 
    outputCol="numerical_features_scaled", 
    withStd=True, 
    withMean=True)
scaler_model = scaler.fit(data)
data = scaler_model.transform(data)
data.show()
# One

**Feature Preperation - Categorical Features** <br>

`String Indexing --> Vector Assembling` <br>

Our categorical features are represented as strings. Which cannot be used as input to ML algorithms in PySpark


**String Indexing** <br>
We need to convert all the string columns to numeric columns.



In [0]:
categorical_columns

In [0]:
categorical_columns_indexed = [name + "_Indexed" for name in categorical_columns]

indexer = StringIndexer(
    inputCols=categorical_columns, 
    outputCols=categorical_columns_indexed)
indexer_model = indexer.fit(data)
data = indexer_model.transform(data)
data.show()


Let's combine all of our categorical features in to one feature vector.

Points to note:
* The `Churn_Indexed` is our target attribute, so it should not be passed
* We can also remove `CustomerID_Indexed` as its just a unique value for each row

In [0]:
categorical_columns_indexed.remove("Churn_Indexed")
categorical_columns_indexed.remove("customerID_Indexed")

In [0]:

categorical_vector_assembler = VectorAssembler(
    inputCols=categorical_columns_indexed, 
    outputCol="categorical_features_vector")

data=categorical_vector_assembler.transform(data)
data.show()

In [0]:
# Now let's combine categorical and numerical feature vectors.

final_vector_assembler = VectorAssembler(
    inputCols=["numerical_features_scaled", "categorical_features_vector"], 
    outputCol="final_feature_vector")

data = final_vector_assembler.transform(data)

data.select("final_feature_vector", "Churn_Indexed").show()

### Task Summary
#### Task Goal
In this task, we preprocessed and cleaned our data by removing the outliers and handling the missing values with mean imputers.

#### Key Takeaways

* To handle missing values in our numerical feature columns, we can use mean, median, or mode imputation. This involves filling in the missing values with the mean, median, or mode of the non-missing values in the same column.

* To remove outliers from our PySpark DataFrame, we can use the filter() method with appropriate filtering conditions to select and remove the rows that contain outliers.

#### **Task 5 - Model Training**
- Train and Test data splitting 
- Creating our model 
- Training our model 
- Make initial predictions using our model

In this task, we are going to start training our model

In [0]:
train, test = data.randomSplit([0.7, 0.3], seed=100)

train.count(), test.count()

In [0]:
# Now let's create and train our desicion tree
dt = DecisionTreeClassifier(
    labelCol="Churn_Indexed", 
    featuresCol="final_feature_vector", 
    maxDepth=3)  # hyper parameter
dt_model = dt.fit(train)


In [0]:
#dt_model.featureImportances
#dt

#Let's make predictions on our test data
predictions_test = dt_model.transform(test)
predictions_test.select("prediction", "Churn").show()


### Task Summary
#### Task Goal
In this task, we prepared features for our binary classification model in PySpark. We combined numerical features using VectorAssembler and scaled them with StandardScaler. For categorical features, we used StringIndexer to convert them to a numerical representation and then combined them with VectorAssembler. Finally, we combined the numerical and categorical feature vectors to create the final feature vector for our decision tree model.

#### Key Takeaways

* To create a Decision Tree in PySpark, you need to combine your numerical and categorical features using VectorAssembler.

* Categorical features should be converted into numerical indices using StringIndexer. This allows them to be used as input for the VectorAssembler.

* Before combining the features, make sure the numerical features are scaled. This can be done using StandardScaler to ensure that they are on the same scale and can be compared fairly by the decision tree algorithm.

### Task Summary
#### Task Goal
In this task, we used PySpark to create a decision tree model to predict customer churn. Firstly, we split our data into training and testing sets. Then, we created our decision tree model and set the hyperparameters based on our knowledge of the data. Finally, we used the model to make our first predictions on the test set.

#### Key Takeaways

* PySpark Decision Tree is a powerful algorithm for binary classification that can be used to predict customer churn in this case.

* Random train-test split is a common approach used to split the data into training and testing sets for model evaluation.

#### **Task 6 - Model Evaluation**
- Calculating area under the ROC curve for the `test` set 
- Calculating area under the ROC curve for the `training` set 
- Hyper parameter tuning